# Notebook 01: Data Loading and Cleaning
**Project:** Parking-Induced Congestion AI System

This notebook handles:
- Loading the raw police violation CSV
- Standardizing column names
- Auto-detecting key columns (lat, lon, datetime, violation type, etc.)
- Cleaning coordinates and text fields
- Extracting time features
- Saving cleaned data to `../data/processed/cleaned_violations.csv`

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully.')

Libraries imported successfully.


## 2. Define Paths and Create Directories

In [2]:
RAW_DATA_PATH   = '../data/raw/jan to may police violation_anonymized791b166.csv'
PROCESSED_DIR   = '../data/processed/'
CHARTS_DIR      = '../data/processed/charts/'
MAPS_DIR        = '../data/processed/maps/'
CLEANED_CSV     = '../data/processed/cleaned_violations.csv'

for d in [PROCESSED_DIR, CHARTS_DIR, MAPS_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'Directory ready: {d}')

Directory ready: ../data/processed/
Directory ready: ../data/processed/charts/
Directory ready: ../data/processed/maps/


## 3. Load CSV with Encoding Fallback

In [3]:
def load_csv_safe(path):
    """Try multiple encodings to load a CSV robustly."""
    encodings = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']
    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            print(f'Loaded successfully with encoding: {enc}')
            return df
        except Exception as e:
            print(f'  Failed with {enc}: {e}')
    raise ValueError(f'Could not load CSV from {path} with any known encoding.')

df_raw = load_csv_safe(RAW_DATA_PATH)
print(f'\nRaw dataset shape: {df_raw.shape}')

Loaded successfully with encoding: utf-8

Raw dataset shape: (298450, 24)


## 4. Initial Inspection

In [4]:
print('=== Shape ===')
print(df_raw.shape)

print('\n=== Columns ===')
print(df_raw.columns.tolist())

print('\n=== Head ===')
display(df_raw.head())

print('\n=== Info ===')
df_raw.info()

=== Shape ===
(298450, 24)

=== Columns ===
['id', 'latitude', 'longitude', 'location', 'vehicle_number', 'vehicle_type', 'description', 'violation_type', 'offence_code', 'created_datetime', 'closed_datetime', 'modified_datetime', 'device_id', 'created_by_id', 'center_code', 'police_station', 'data_sent_to_scita', 'junction_name', 'action_taken_timestamp', 'data_sent_to_scita_timestamp', 'updated_vehicle_number', 'updated_vehicle_type', 'validation_status', 'validation_timestamp']

=== Head ===


,id,latitude,longitude,location,vehicle_number,vehicle_type,description,violation_type,offence_code,created_datetime,...,center_code,police_station,data_sent_to_scita,junction_name,action_taken_timestamp,data_sent_to_scita_timestamp,updated_vehicle_number,updated_vehicle_type,validation_status,validation_timestamp
0,FKID000000,12.925557,77.618665,"18th Main Road, Block 2, Koramangala, Bengalur...",FKN00GL0000,CAR,NaN,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00,...,9.0,Madiwala,True,No Junction,NaN,NaN,FKN00GL0000,MAXI-CAB,approved,2023-11-30 03:08:24.818+00
1,FKID000001,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony...",FKN00GL0001,CAR,NaN,"[""NO PARKING""]",[113],2023-11-24 22:46:46+00,...,82.0,Bellandur,False,No Junction,NaN,NaN,NaN,NaN,NaN,NaN
2,FKID000002,12.925449,77.618504,"Koramangala 2nd Block, Kormangala West, Bengal...",FKN00GL0002,CAR,NaN,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]","[112,107]",2023-11-20 00:27:46+00,...,9.0,Madiwala,True,No Junction,NaN,NaN,FKN00GL0002,MAXI-CAB,approved,2023-11-30 03:08:56.998+00
3,FKID000003,12.956521,77.518618,"6th Cross Road, Manasa Layout, Nagarbhavi, Ben...",FKN00GL0003,SCOOTER,NaN,"[""NO PARKING""]",[113],2023-11-16 06:47:46+00,...,26.0,Byatarayanapura,True,No Junction,NaN,NaN,FKN00GL0003,SCOOTER,approved,2023-11-18 23:35:12.428+00
4,FKID000004,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",FKN00GL0004,TANKER,NaN,"[""NO PARKING""]",[113],2023-11-22 04:56:46+00,...,3.0,Upparpet,True,BTP044 - Sagar Theatre Junction,NaN,NaN,FKN00GL0004,TANKER,approved,2023-11-30 03:11:32.796+00



=== Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 298450 entries, 0 to 298449
Data columns (total 24 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            298450 non-null  object 
 1   latitude                      298450 non-null  float64
 2   longitude                     298450 non-null  float64
 3   location                      295409 non-null  object 
 4   vehicle_number                298450 non-null  object 
 5   vehicle_type                  298450 non-null  object 
 6   description                   0 non-null       float64
 7   violation_type                298450 non-null  object 
 8   offence_code                  298450 non-null  object 
 9   created_datetime              298450 non-null  object 
 10  closed_datetime               0 non-null       float64
 11  modified_datetime             298450 non-null  object 
 12  device_id                     

In [5]:
print('\n=== Missing Values ===')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_count', ascending=False)
display(missing_df)


=== Missing Values ===


,missing_count,missing_pct
description,298450,100.00
closed_datetime,298450,100.00
action_taken_timestamp,298450,100.00
data_sent_to_scita_timestamp,256289,85.87
updated_vehicle_type,125254,41.97
validation_status,125254,41.97
validation_timestamp,125254,41.97
updated_vehicle_number,125254,41.97
center_code,11260,3.77
location,3041,1.02


## 5. Standardize Column Names to snake_case

In [6]:
def to_snake_case(col):
    """Convert column name to snake_case."""
    import re
    col = str(col).strip()
    col = re.sub(r'[\s\-\/\.\(\)]+', '_', col)
    col = re.sub(r'[^a-zA-Z0-9_]', '', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_').lower()
    return col

df = df_raw.copy()
df.columns = [to_snake_case(c) for c in df.columns]

print('Standardized column names:')
print(df.columns.tolist())

Standardized column names:
['id', 'latitude', 'longitude', 'location', 'vehicle_number', 'vehicle_type', 'description', 'violation_type', 'offence_code', 'created_datetime', 'closed_datetime', 'modified_datetime', 'device_id', 'created_by_id', 'center_code', 'police_station', 'data_sent_to_scita', 'junction_name', 'action_taken_timestamp', 'data_sent_to_scita_timestamp', 'updated_vehicle_number', 'updated_vehicle_type', 'validation_status', 'validation_timestamp']


## 6. Auto-Detect Key Columns

In [7]:
def find_column(df, keywords, dtype_check=None):
    """
    Find the best matching column by scanning column names for keywords.
    Optionally filter by dtype substring (e.g. 'float', 'int', 'object').
    Returns the first match or None.
    """
    cols = df.columns.tolist()
    for kw in keywords:
        for c in cols:
            if kw in c.lower():
                if dtype_check:
                    if dtype_check in str(df[c].dtype):
                        return c
                else:
                    return c
    return None

# Latitude
lat_col = find_column(df, ['lat', 'latitude', 'y_coord', 'y coord', 'lat_', '_lat'])
# Longitude
lon_col = find_column(df, ['lon', 'lng', 'long', 'longitude', 'x_coord', 'x coord', 'lon_', '_lon'])
# Datetime
dt_col  = find_column(df, ['datetime', 'timestamp', 'date_time', 'incident_date', 'offence_date',
                            'violation_date', 'challan_date', 'date'])
# Separate time column (in case date and time are split)
time_col = find_column(df, ['time', 'offence_time', 'violation_time', 'challan_time'])
# Violation type
viol_col = find_column(df, ['violation', 'offence', 'offense', 'violation_type', 'offence_type',
                             'challan_type', 'charge', 'category', 'description'])
# Police station
ps_col   = find_column(df, ['police_station', 'station', 'ps_name', 'police', 'ward', 'zone',
                             'district', 'unit'])
# Location / Junction
loc_col  = find_column(df, ['location', 'junction', 'place', 'spot', 'road', 'street',
                             'address', 'area', 'locality'])
# Vehicle type
veh_col  = find_column(df, ['vehicle', 'vehicle_type', 'vehicle_category', 'veh_type',
                             'car_type', 'transport'])

print('=== Auto-Detected Columns ===')
print(f'  Latitude        : {lat_col}')
print(f'  Longitude       : {lon_col}')
print(f'  Datetime        : {dt_col}')
print(f'  Time (separate) : {time_col}')
print(f'  Violation type  : {viol_col}')
print(f'  Police station  : {ps_col}')
print(f'  Location        : {loc_col}')
print(f'  Vehicle type    : {veh_col}')

=== Auto-Detected Columns ===
  Latitude        : latitude
  Longitude       : longitude
  Datetime        : created_datetime
  Time (separate) : created_datetime
  Violation type  : violation_type
  Police station  : police_station
  Location        : location
  Vehicle type    : vehicle_number


## 7. Parse and Combine Date/Time Columns

In [8]:
def parse_datetime_column(df, dt_col, time_col=None):
    """
    Parse a datetime column. If date and time are separate, combine them.
    Returns the DataFrame with a 'datetime' canonical column added.
    """
    if dt_col is None:
        print('WARNING: No date/time column detected. Skipping datetime parsing.')
        df['datetime'] = pd.NaT
        return df, 'none'

    if time_col and time_col != dt_col:
        # Try combining date + time strings
        try:
            combined = df[dt_col].astype(str).str.strip() + ' ' + df[time_col].astype(str).str.strip()
            df['datetime'] = pd.to_datetime(combined, infer_datetime_format=True, errors='coerce')
            ok = df['datetime'].notna().sum()
            print(f'Combined {dt_col} + {time_col} -> datetime. Valid: {ok}/{len(df)}')
            if ok / len(df) < 0.5:
                raise ValueError('Too many NaT after combining')
            return df, f'{dt_col} + {time_col}'
        except Exception as e:
            print(f'  Combine failed: {e}. Falling back to date-only parsing.')

    df['datetime'] = pd.to_datetime(df[dt_col], infer_datetime_format=True, errors='coerce')
    ok = df['datetime'].notna().sum()
    print(f'Parsed {dt_col} -> datetime. Valid: {ok}/{len(df)}')
    return df, dt_col

df, dt_source = parse_datetime_column(df, dt_col, time_col)

Parsed created_datetime -> datetime. Valid: 298445/298450


## 8. Extract Time Features

In [9]:
def extract_time_features(df):
    """Extract hour, day, month, day_of_week, weekday_name, is_weekend from 'datetime' column."""
    if 'datetime' not in df.columns or df['datetime'].isna().all():
        print('WARNING: datetime column missing/all-null. Setting time feature columns to NaN.')
        for col in ['hour','day','month','day_of_week','weekday_name','is_weekend']:
            df[col] = np.nan
        return df

    df['hour']         = df['datetime'].dt.hour
    df['day']          = df['datetime'].dt.day
    df['month']        = df['datetime'].dt.month
    df['day_of_week']  = df['datetime'].dt.dayofweek          # 0=Mon, 6=Sun
    df['weekday_name'] = df['datetime'].dt.day_name()
    df['is_weekend']   = df['day_of_week'].isin([5, 6]).astype(int)
    print('Time features extracted: hour, day, month, day_of_week, weekday_name, is_weekend')
    return df

df = extract_time_features(df)
display(df[['datetime','hour','day','month','weekday_name','is_weekend']].head())

Time features extracted: hour, day, month, day_of_week, weekday_name, is_weekend


,datetime,hour,day,month,weekday_name,is_weekend
0,2023-11-20 00:28:46+00:00,0.0,20.0,11.0,Monday,0
1,2023-11-24 22:46:46+00:00,22.0,24.0,11.0,Friday,0
2,2023-11-20 00:27:46+00:00,0.0,20.0,11.0,Monday,0
3,2023-11-16 06:47:46+00:00,6.0,16.0,11.0,Thursday,0
4,2023-11-22 04:56:46+00:00,4.0,22.0,11.0,Wednesday,0


## 9. Clean Latitude and Longitude

In [10]:
def clean_coordinates(df, lat_col, lon_col):
    """
    - Convert lat/lon to numeric
    - Remove rows where lat/lon are NaN
    - Remove (0, 0) placeholder rows
    - Remove coordinates outside valid global range
    Returns cleaned df with canonical 'latitude' and 'longitude' columns.
    """
    if lat_col is None or lon_col is None:
        print('WARNING: Lat/Lon columns not detected. Skipping coordinate cleaning.')
        df['latitude']  = np.nan
        df['longitude'] = np.nan
        return df

    df['latitude']  = pd.to_numeric(df[lat_col],  errors='coerce')
    df['longitude'] = pd.to_numeric(df[lon_col], errors='coerce')

    before = len(df)
    df = df.dropna(subset=['latitude', 'longitude'])
    print(f'Rows after removing NaN coords: {len(df)} (removed {before - len(df)})')

    before = len(df)
    df = df[~((df['latitude'] == 0) & (df['longitude'] == 0))]
    print(f'Rows after removing (0,0): {len(df)} (removed {before - len(df)})')

    before = len(df)
    df = df[
        (df['latitude']  >= -90)  & (df['latitude']  <= 90) &
        (df['longitude'] >= -180) & (df['longitude'] <= 180)
    ]
    print(f'Rows after range filter: {len(df)} (removed {before - len(df)})')
    return df

df = clean_coordinates(df, lat_col, lon_col)
print(f'Coordinate range -> Lat: {df["latitude"].min():.4f} to {df["latitude"].max():.4f}')
print(f'                    Lon: {df["longitude"].min():.4f} to {df["longitude"].max():.4f}')

Rows after removing NaN coords: 298450 (removed 0)
Rows after removing (0,0): 298450 (removed 0)
Rows after range filter: 298450 (removed 0)
Coordinate range -> Lat: 12.8027 to 13.2937
                    Lon: 77.4426 to 77.7717


## 10. Rename Detected Columns to Canonical Names

In [11]:
def rename_to_canonical(df, viol_col, ps_col, loc_col, veh_col):
    """Rename detected columns to canonical names if not already named that way."""
    rename_map = {}
    if viol_col and viol_col != 'violation_type':
        rename_map[viol_col] = 'violation_type'
    if ps_col and ps_col != 'police_station':
        rename_map[ps_col] = 'police_station'
    if loc_col and loc_col != 'location_or_junction':
        rename_map[loc_col] = 'location_or_junction'
    if veh_col and veh_col != 'vehicle_type':
        rename_map[veh_col] = 'vehicle_type'

    if rename_map:
        df = df.rename(columns=rename_map)
        print(f'Renamed columns: {rename_map}')
    else:
        print('No renaming needed (columns already have canonical names or were not detected).')
    return df

df = rename_to_canonical(df, viol_col, ps_col, loc_col, veh_col)

# Add missing canonical columns as NaN so downstream notebooks don't crash
for canon in ['violation_type', 'police_station', 'location_or_junction', 'vehicle_type']:
    if canon not in df.columns:
        df[canon] = np.nan
        print(f'Added missing canonical column as NaN: {canon}')

print('\nCanonical columns present:', [c for c in ['latitude','longitude','datetime','violation_type',
      'police_station','location_or_junction','vehicle_type'] if c in df.columns])

Renamed columns: {'location': 'location_or_junction', 'vehicle_number': 'vehicle_type'}

Canonical columns present: ['latitude', 'longitude', 'datetime', 'violation_type', 'police_station', 'location_or_junction', 'vehicle_type']


## 11. Clean Text Columns

In [16]:
def clean_text_column(series):
    """Strip whitespace, title-case, replace blanks with NaN."""
    return series.astype(str).str.strip().str.title().replace({'Nan': np.nan, 'None': np.nan, '': np.nan})

text_cols = ['violation_type', 'police_station', 'location_or_junction']
for col in text_cols:
    if col in df.columns:
        df[col] = clean_text_column(df[col])
        print(f'Cleaned text column: {col} | unique values: {df[col].nunique()}')

Cleaned text column: violation_type | unique values: 991
Cleaned text column: police_station | unique values: 54
Cleaned text column: location_or_junction | unique values: 10941


## 12. Remove Duplicate Rows

In [13]:
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df)} duplicate rows. Remaining: {len(df)}')

Removed 0 duplicate rows. Remaining: 298450


## 13. Save Cleaned Data

In [14]:
df.to_csv(CLEANED_CSV, index=False)
print(f'Cleaned data saved to: {CLEANED_CSV}')
print(f'Shape: {df.shape}')
display(df.head())

Cleaned data saved to: ../data/processed/cleaned_violations.csv
Shape: (298450, 31)


,id,latitude,longitude,location_or_junction,vehicle_type,vehicle_type,description,violation_type,offence_code,created_datetime,...,updated_vehicle_type,validation_status,validation_timestamp,datetime,hour,day,month,day_of_week,weekday_name,is_weekend
0,FKID000000,12.925557,77.618665,"18Th Main Road, Block 2, Koramangala, Bengalur...",FKN00GL0000,CAR,NaN,"[""Wrong Parking"",""Parking Near Road Crossing""]","[112,104]",2023-11-20 00:28:46+00,...,MAXI-CAB,approved,2023-11-30 03:08:24.818+00,2023-11-20 00:28:46+00:00,0.0,20.0,11.0,0.0,Monday,0
1,FKID000001,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony...",FKN00GL0001,CAR,NaN,"[""No Parking""]",[113],2023-11-24 22:46:46+00,...,NaN,NaN,NaN,2023-11-24 22:46:46+00:00,22.0,24.0,11.0,4.0,Friday,0
2,FKID000002,12.925449,77.618504,"Koramangala 2Nd Block, Kormangala West, Bengal...",FKN00GL0002,CAR,NaN,"[""Wrong Parking"",""Parking In A Main Road""]","[112,107]",2023-11-20 00:27:46+00,...,MAXI-CAB,approved,2023-11-30 03:08:56.998+00,2023-11-20 00:27:46+00:00,0.0,20.0,11.0,0.0,Monday,0
3,FKID000003,12.956521,77.518618,"6Th Cross Road, Manasa Layout, Nagarbhavi, Ben...",FKN00GL0003,SCOOTER,NaN,"[""No Parking""]",[113],2023-11-16 06:47:46+00,...,SCOOTER,approved,2023-11-18 23:35:12.428+00,2023-11-16 06:47:46+00:00,6.0,16.0,11.0,3.0,Thursday,0
4,FKID000004,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",FKN00GL0004,TANKER,NaN,"[""No Parking""]",[113],2023-11-22 04:56:46+00,...,TANKER,approved,2023-11-30 03:11:32.796+00,2023-11-22 04:56:46+00:00,4.0,22.0,11.0,2.0,Wednesday,0


## 14. Summary Table

In [15]:
raw_rows     = len(df_raw)
cleaned_rows = len(df)
removed_rows = raw_rows - cleaned_rows

summary = pd.DataFrame([
    {'Metric': 'Raw rows',                         'Value': raw_rows},
    {'Metric': 'Cleaned rows',                     'Value': cleaned_rows},
    {'Metric': 'Rows removed',                     'Value': removed_rows},
    {'Metric': 'Detected latitude column',         'Value': lat_col},
    {'Metric': 'Detected longitude column',        'Value': lon_col},
    {'Metric': 'Detected datetime source',         'Value': dt_source},
    {'Metric': 'Detected violation column',        'Value': viol_col},
    {'Metric': 'Detected police station column',   'Value': ps_col},
    {'Metric': 'Detected location column',         'Value': loc_col},
    {'Metric': 'Detected vehicle type column',     'Value': veh_col},
])

print('=== Data Cleaning Summary ===')
display(summary)

=== Data Cleaning Summary ===


,Metric,Value
0,Raw rows,298450
1,Cleaned rows,298450
2,Rows removed,0
3,Detected latitude column,latitude
4,Detected longitude column,longitude
5,Detected datetime source,created_datetime
6,Detected violation column,violation_type
7,Detected police station column,police_station
8,Detected location column,location
9,Detected vehicle type column,vehicle_number
